# Batch validation of existing Experiment 2 reuse plans

Checks the saved fixed-active-width plans using the **generator-independent** checker (`reuse_validity.py`). This notebook does not run routing or regenerate reuse plans. The check is static DAG admissibility, **not** proof of full dynamic-circuit semantics.

In [1]:
from pathlib import Path
import ast
import pandas as pd
from qiskit import qasm2
from reuse_validity import ReuseValidityChecker

# Set this to the directory containing system9.qasm, sym9_146.qasm, etc.
BENCH_DIR = Path(r"C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main\benchmarks")
PLAN_CSV = Path("experiment2_results/experiment2_plan_features.csv")
OUT_DIR = Path("validity_checker_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert BENCH_DIR.is_dir(), f"Set BENCH_DIR to your local QASM directory: {BENCH_DIR}"
assert PLAN_CSV.is_file(), f"Saved Experiment 2 results not found: {PLAN_CSV}"
plans = pd.read_csv(PLAN_CSV)
print("Plans:", len(plans), "Circuits:", plans['circuit'].nunique())

Plans: 50 Circuits: 5


In [2]:
def parse_chains(raw):
    value = ast.literal_eval(raw) if isinstance(raw, str) else raw
    return [tuple(int(q) for q in chain) for chain in value]

all_results = []
for circuit_name, rows in plans.groupby('circuit', sort=True):
    circuit_path = BENCH_DIR / circuit_name
    assert circuit_path.is_file(), f"Original QASM circuit not found: {circuit_path}"
    qc = qasm2.load(str(circuit_path))
    checker = ReuseValidityChecker(qc)  # build/cache DAGs once per circuit
    for _, row in rows.iterrows():
        active_chains = parse_chains(row['reuse_chains'])
        res = checker.validate(active_chains, target_width=int(row['width']))
        all_results.append({
            'circuit': circuit_name,
            'plan_id': row['plan_id'],
            'gidnet_seed': row.get('gidnet_seed'),
            'saved_width': int(row['width']),
            'is_valid_static': res.is_valid,
            'reason': res.reason,
            'actual_active_width': res.active_width,
            'full_width_with_inactive_singletons': res.full_width,
            'num_reuse_boundaries': res.num_reuse_boundaries,
            'details': repr(dict(res.details)),
        })

out = pd.DataFrame(all_results)
output_path = OUT_DIR / 'validation_of_experiment2_saved_plans.csv'
out.to_csv(output_path, index=False)
print("Saved:", output_path)
print(out.groupby(['circuit', 'reason']).size().to_string())
print("Static valid:", int(out['is_valid_static'].sum()), '/', len(out))
display(out.loc[~out['is_valid_static']])

Saved: validity_checker_results\validation_of_experiment2_saved_plans.csv
circuit            reason   
alu-bdd_288.qasm   ok_static     6
mini_alu_305.qasm  ok_static    16
rd73_140.qasm      ok_static     6
sym9_146.qasm      ok_static     9
system9.qasm       ok_static    13
Static valid: 50 / 50


,circuit,plan_id,gidnet_seed,saved_width,is_valid_static,reason,actual_active_width,full_width_with_inactive_singletons,num_reuse_boundaries,details


## Interpretation

All the saved Experiment 2 plans are expected to be valid under the original Experiment 3 checker. If any fails here, **do not assume the old data is wrong**: inspect whether this is an actual invalid reuse relation or a difference in DAG construction, source-circuit support, inactive-qubit handling, or the GidNET candidate matrix convention. Fix that discrepancy before using this checker for new operators.

**Next**: run `python -m pytest -q test_reuse_validity.py`, then optionally perform compilation/reset-insertion and simulation checks of selected valid/invalid test cases.